# Comparing clustering methods: how they reach different solutions

Give six clustering methods the **same days** and ask each for the **same number of
typical periods**, and they will not agree. Each method optimises a *different idea of a
good grouping*, so each carves the data differently — and those differences flow straight
through to the typical periods your downstream model sees.

This tutorial makes that visible on **12 hand-designed days**, small enough to follow every
period by eye. By the end you will be able to:

1. **predict** which methods group by *value* and which group by *calendar position*;
2. **explain** why `kmeans`, `kmedoids` and `hierarchical` agree on the partition here, yet
   `kmaxoids` splits off the storm and `averaging`/`contiguous` ignore similarity;
3. **read** the consequence in the typical periods — a synthetic mean vs. a real day vs. a
   preserved extreme — and in the reconstruction error.

This is the *learning-oriented* companion to the task-oriented
[Clustering methods how-to](../how-to/clustering_methods.ipynb) (which benchmarks accuracy and
speed on a real dataset) and to the [Partitional clustering explanation](../explanation/how-it-works/02_clustering/01_partitional_clustering.ipynb)
(which derives the partitional objectives). Here we focus on a single question: *why do they
disagree?*

> Throughout we pass `preserve_column_means=False` so that each typical period is **exactly**
> its cluster's mean, medoid or maxoid — nothing is adjusted afterwards. Mean-rescaling is a
> separate step, covered in [Rescaling](../explanation/how-it-works/05_rescaling.ipynb).

## 1  The data: 12 designed days

The sample is built so the lesson is unambiguous. There are four kinds of day:

| archetype | solar | load | how many |
|---|---|---|---|
| **sunny** | bright midday peak | low | 5 |
| **cloudy** | dim | medium | 4 |
| **high-load** | dim | high | 2 |
| **storm** | none | **extreme** load peak | 1 |

Two design choices do all the work:

* the single **storm** day is a clear outlier — far from every other day in value space;
* the days are **interleaved in the calendar** (sunny, cloudy, high-load, …) so that
  *position on the calendar tells you nothing about a day's shape*.

The shared sample lives in `../data/comparison_days.csv`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

import tsam
from tsam import ClusterConfig

pio.renderers.default = "notebook_connected"

# 12 designed days, 6-hourly, two attributes (solar, load).
days = pd.read_csv("../data/comparison_days.csv", index_col=0, parse_dates=True)
N_DAYS, N_T = 12, 4

# What each day "really" is, by construction — used only for labels/colours.
archetype = [
    "sunny",
    "cloudy",
    "high-load",
    "sunny",
    "cloudy",
    "storm",
    "cloudy",
    "sunny",
    "high-load",
    "sunny",
    "cloudy",
    "sunny",
]

print(
    days.shape,
    "->",
    N_DAYS,
    "days x",
    N_T,
    "six-hourly steps x",
    len(days.columns),
    "attributes",
)
days.head(8)

In [ ]:
# The two attributes across all 12 days. The storm is the load spike on day 5.
long = days.reset_index(names="time").melt(
    id_vars="time", var_name="attribute", value_name="value"
)
fig = px.line(
    long,
    x="time",
    y="value",
    facet_row="attribute",
    title="The 12 designed days — note the storm spike in load (day 5)",
)
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

## 2  Run all six methods at k = 3

We ask each method for **3 typical days** and line up the resulting partitions. To compare
groupings we *canonicalise* the labels (rename clusters by order of first appearance), so two
methods that group the days the same way show identical numbers.

In [ ]:
# Relabel clusters by first appearance so identical groupings look identical.
def canon(labels):
    remap, out = {}, []
    for x in labels:
        remap.setdefault(x, len(remap))
        out.append(remap[x])
    return np.array(out)


methods = ["kmeans", "kmedoids", "hierarchical", "kmaxoids", "contiguous", "averaging"]
results, rows = {}, []
for m in methods:
    np.random.seed(0)
    r = tsam.aggregate(
        days,
        n_clusters=3,
        period_duration="1D",
        cluster=ClusterConfig(method=m),
        preserve_column_means=False,
    )
    results[m] = r
    rows.append(
        {
            "method": m,
            "partition (day -> cluster)": str(canon(r.cluster_assignments)),
            "weighted RMSE": round(r.accuracy.weighted_rmse, 3),
        }
    )
pd.DataFrame(rows)

Six methods, but only **four distinct partitions**:

| partition | methods | in words |
|---|---|---|
| `0 1 2 0 1 2 1 0 2 0 1 0` | **kmeans, kmedoids, hierarchical** | the three *value* groups; **storm absorbed** into high-load |
| `0 1 1 0 1 2 1 0 1 0 1 0` | **kmaxoids** | **storm isolated** as its own cluster; cloudy + high-load merged |
| `0 0 0 0 0 1 2 2 2 2 2 2` | **contiguous** | value-aware but **calendar-blocked**: storm singleton between two runs |
| `0 0 0 0 1 1 1 1 2 2 2 2` | **averaging** | three **equal calendar blocks**; values ignored |

The rest of the tutorial explains *why* each of these arises. The three feature-based
J-minimisers agree here only because the value groups are well separated; the interesting
question is why the other three diverge.

## 3  The map: the 12 days in feature space

Clustering does not see day-shapes; it sees each day as a single **point** in an 8-D space
(2 attributes x 4 timesteps), after min–max normalisation. Projecting those points to 2-D
with PCA gives a faithful map — here it keeps ~99% of the variance, so distances on the plot
are essentially the distances the algorithms use.

In [ ]:
# Reproduce the feature space tsam clusters in: min-max normalise each attribute,
# then unstack each day into one 8-D vector (solar t0..t3, load t0..t3).
norm = (days - days.min()) / (days.max() - days.min())
P = (
    norm.values.reshape(N_DAYS, N_T, len(days.columns))
    .transpose(0, 2, 1)
    .reshape(N_DAYS, -1)
)

# Project the 8-D day-vectors to 2-D with PCA (numpy SVD).
Dc = P - P.mean(axis=0)
_u, _s, _vt = np.linalg.svd(Dc, full_matrices=False)
pts = Dc @ _vt[:2].T
explained = (_s**2 / (_s**2).sum())[:2].sum()

palette = {
    "sunny": "#F2B705",
    "cloudy": "#7F8C9A",
    "high-load": "#1F77B4",
    "storm": "#D62728",
}
fig = px.scatter(
    x=pts[:, 0],
    y=pts[:, 1],
    color=archetype,
    color_discrete_map=palette,
    text=[f"day_{i}" for i in range(N_DAYS)],
    title=f"The 12 days in feature space (2-D PCA, {explained:.0%} of variance)",
)
fig.update_traces(textposition="top center", marker={"size": 13})
fig.update_layout(legend_title_text="archetype", xaxis_title="PC 1", yaxis_title="PC 2")
fig.show()

Read the map: the **sunny** days cluster top-left, **cloudy** in the middle, **high-load**
lower, and the lone **storm** (day_5, red) sits far out on its own. Every method below is
just a different rule for covering these points with three centres.

## 4  Divergence I — minimise distance vs. maximise spread

The first split in behaviour is about the **objective**.

* **k-means / k-medoids / hierarchical** all drive down the *total distance from each day to
  its cluster centre*. With three centres to spend, the cheapest way to do that is to cover
  the three **dense** groups — and let the lone storm join the nearest one (high-load). The
  storm is one day; absorbing it barely moves the total.
* **k-maxoids** does the opposite: it picks centres to be **far apart**. The storm is the
  farthest point from everything, so spending a centre on it *maximises* spread. That forces
  the remaining two centres to cover cloudy **and** high-load together.

The two panels below show the same 12 points. Lines connect each day to the centre it is
assigned to; on the left the centre is the cluster mean (★), on the right it is a real day
chosen for spread (ringed).

In [ ]:
def cluster_colors(labels):
    base = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]
    lab = canon(labels)
    return [base[c] for c in lab], lab


fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Minimise distance — kmeans = kmedoids = hierarchical",
        "Maximise spread — kmaxoids",
    ),
)

# Left: shared J-partition with k-means centroids (★).
asg = np.asarray(results["kmeans"].cluster_assignments)
cols, lab = cluster_colors(asg)
centroids = np.array([P[asg == c].mean(0) for c in np.unique(asg)])
cent2d = (centroids - P.mean(0)) @ _vt[:2].T
for p in range(N_DAYS):
    cx, cy = cent2d[lab[p]]
    fig.add_trace(
        go.Scatter(
            x=[pts[p, 0], cx],
            y=[pts[p, 1], cy],
            mode="lines",
            line={"color": cols[p], "width": 1},
            opacity=0.4,
            showlegend=False,
        ),
        1,
        1,
    )
fig.add_trace(
    go.Scatter(
        x=pts[:, 0],
        y=pts[:, 1],
        mode="markers+text",
        text=[f"day_{p}" for p in range(N_DAYS)],
        textposition="top center",
        marker={"size": 11, "color": cols},
        showlegend=False,
    ),
    1,
    1,
)
fig.add_trace(
    go.Scatter(
        x=cent2d[:, 0],
        y=cent2d[:, 1],
        mode="markers",
        marker={"symbol": "star", "size": 18, "color": "black"},
        showlegend=False,
    ),
    1,
    1,
)

# Right: k-maxoids partition with its real-period centres (ringed).
asg = np.asarray(results["kmaxoids"].cluster_assignments)
cols, lab = cluster_colors(asg)
centers_idx = list(results["kmaxoids"].clustering.cluster_centers)
center_of_label = {lab[ci]: ci for ci in centers_idx}
for p in range(N_DAYS):
    ci = center_of_label[lab[p]]
    fig.add_trace(
        go.Scatter(
            x=[pts[p, 0], pts[ci, 0]],
            y=[pts[p, 1], pts[ci, 1]],
            mode="lines",
            line={"color": cols[p], "width": 1},
            opacity=0.4,
            showlegend=False,
        ),
        1,
        2,
    )
fig.add_trace(
    go.Scatter(
        x=pts[:, 0],
        y=pts[:, 1],
        mode="markers+text",
        text=[f"day_{p}" for p in range(N_DAYS)],
        textposition="top center",
        marker={"size": 11, "color": cols},
        showlegend=False,
    ),
    1,
    2,
)
fig.add_trace(
    go.Scatter(
        x=pts[centers_idx, 0],
        y=pts[centers_idx, 1],
        mode="markers",
        marker={
            "symbol": "circle-open",
            "size": 22,
            "line": {"color": "black", "width": 3},
        },
        showlegend=False,
    ),
    1,
    2,
)
fig.update_layout(
    height=470,
    width=950,
    title="Same 12 days — storm (day_5) absorbed (left) or made its own centre (right)",
)
fig.update_xaxes(title_text="PC 1")
fig.update_yaxes(title_text="PC 2", col=1)
fig.show()

The price of each choice shows in the error: k-maxoids' weighted RMSE is the highest of the
feature-based methods, because merging cloudy with high-load reconstructs ten ordinary days
worse in exchange for nailing one extreme. Whether that trade is worth it depends on your
model — if a single peak drives investment decisions, capturing the storm may matter more
than average fit. See [Extreme periods](../how-to/extreme_periods.ipynb) for the
targeted alternative: keeping the storm *without* paying for it across the other ten days.

## 5  Divergence II — same partition, different representative

k-means, k-medoids and hierarchical produced the **identical partition** above, so how can
they differ at all? Through the **representative** — the single profile that stands in for
each cluster. Look at the cluster that holds the storm (its members are the two high-load
days and the storm):

* **k-means** uses the **mean** — a synthetic profile that lies *between* the real days and
  flattens the storm peak;
* **k-medoids / hierarchical** use the **medoid** — the most central *real* day, which here
  is a high-load day, so it is a genuine profile but still misses the storm;
* **k-maxoids** isolated the storm, so *its* representative for that day **is the storm** —
  the peak survives intact.

In [ ]:
# The representative profile of whichever cluster `day` was assigned to.
def rep_for_day(r, day=5, col="load"):
    cl = np.asarray(r.cluster_assignments)[day]
    return r.cluster_representatives.loc[cl, col].values


steps = list(range(N_T))
load_by_day = days["load"].values.reshape(N_DAYS, N_T)
asg = np.asarray(results["kmeans"].cluster_assignments)
storm_cluster = asg[5]
members = [i for i in range(N_DAYS) if asg[i] == storm_cluster]

fig = go.Figure()
for p in members:  # the real member days (grey)
    fig.add_trace(
        go.Scatter(
            x=steps,
            y=load_by_day[p],
            mode="lines",
            line={"color": "lightgray", "width": 1},
            name=f"day_{p} ({archetype[p]})",
        )
    )
for m, color in [
    ("kmeans", "#636EFA"),
    ("kmedoids", "#00CC96"),
    ("kmaxoids", "#D62728"),
]:
    fig.add_trace(
        go.Scatter(
            x=steps,
            y=rep_for_day(results[m]),
            mode="lines+markers",
            line={"color": color, "width": 3},
            name=f"{m} representative",
        )
    )
fig.update_layout(
    title="Representative of the storm's cluster (load) — mean vs. real day vs. preserved extreme",
    xaxis_title="timestep",
    yaxis_title="load (normalised)",
    height=440,
    width=780,
)
fig.show()

The grey lines are the real member days (one peaks at the storm's 1.0). The **k-medoids**
line sits exactly on a real high-load day; the **k-means** line floats between the members;
the **k-maxoids** line rides the storm. Same data, three philosophies of "what should a
typical day look like". This representative axis is explored fully in
[Representation](../explanation/how-it-works/03_representation.ipynb).

## 6  Divergence III — grouping by calendar position

The last two methods never look at value similarity at all; they group by **position on the
calendar**. Because our days are interleaved, this lands them far from the value-based
partition.

* **averaging** cuts the timeline into three **equal blocks** (days 0–3, 4–7, 8–11),
  whatever sits inside;
* **contiguous** is value-*aware* but may only merge **adjacent** days, so it is forced into
  runs — here it strands the storm as a singleton block between two long runs.

Both keep calendar order; neither can put the scattered sunny days together.

In [ ]:
results["contiguous"].plot.clusters_over_time(
    columns=["load"], title="Contiguous — value-aware, but only adjacent days may merge"
)

In [ ]:
results["averaging"].plot.clusters_over_time(
    columns=["load"], title="Averaging — three equal calendar blocks, values ignored"
)

A subtlety worth noting: on this adversarial ordering **contiguous is actually the *worst*
of the six**, slightly behind blind averaging. Being value-aware does not help when the
calendar constraint forces a bad split — a reminder that `contiguous`/`averaging` are for
when calendar order *must* be preserved (e.g. for seasonal storage), not for best fit. The
formulation of these methods is covered in
[Agglomerative clustering](../explanation/how-it-works/02_clustering/02_agglomerative_clustering.ipynb) and
[Averaging](../explanation/how-it-works/02_clustering/04_averaging.ipynb).

## 7  The consequence: reconstruction

"Different partition" is abstract; the reconstructed load makes it concrete. Each real day is
replaced by its cluster's representative and stitched back onto the calendar. Watch the storm
(day 5) and the interleaving:

In [ ]:
frames = [
    pd.DataFrame(
        {"time": days.index, "load": days["load"].values, "series": "original"}
    )
]
for m in ["kmeans", "kmaxoids", "averaging"]:
    s = results[m].reconstructed["load"]
    frames.append(pd.DataFrame({"time": s.index, "load": s.values, "series": m}))
px.line(
    pd.concat(frames),
    x="time",
    y="load",
    color="series",
    title="Load reconstruction — kmeans tracks shape, kmaxoids spikes the storm, "
    "averaging is blocky",
).show()

**k-means** follows the day-to-day shape but clips the storm peak; **k-maxoids** is the only
one to reach the storm's height (it kept a centre there) but is coarser elsewhere;
**averaging** is blocky and value-blind, missing both the shape and the peak.

## 8  Recap — which logic, when

| method | grouping logic | typical representative | reach for it when… |
|---|---|---|---|
| **kmeans** | minimise distance to centroids | synthetic mean | you want the best average fit, fast |
| **kmedoids** | minimise distance, centres are real days | a real day | representatives must be physically real |
| **hierarchical** | merge nearest groups bottom-up | a real day | you want a robust, deterministic default |
| **kmaxoids** | maximise spread between centres | an extreme day | extremes matter more than average fit |
| **contiguous** | merge nearest *adjacent* groups | a real day | calendar order must be preserved |
| **averaging** | equal calendar blocks | block mean | you need the simplest possible baseline |

The headline: **what a method optimises decides what it keeps.** Distance-minimisers protect
the average and quietly drop extremes; spread-maximisers do the reverse; calendar methods
preserve order at the expense of similarity.

Note the last column is only half the decision. Every representative above is the method's
*default* — and the representative is an independent lever, so "kmeans gives you a synthetic
mean" is a default, not a law. Choosing across all the levers at once is the subject of
[Choosing a method](choosing_a_method.ipynb).

**Where to go next**

- [Choosing a method](choosing_a_method.ipynb) — this decision plus the other three levers:
  representation, extremes and segmentation.
- [Clustering methods how-to](../how-to/clustering_methods.ipynb) — accuracy and speed of all
  six on a real six-week dataset.
- [Partitional clustering](../explanation/how-it-works/02_clustering/01_partitional_clustering.ipynb) — how k-means and
  k-medoids are formulated and solved.
- [Extremal-prototype selection](../explanation/how-it-works/02_clustering/03_extremal_prototype_selection.ipynb) — how
  k-maxoids maximises spread between representatives instead.
- [Representations how-to](../how-to/representations.ipynb) — choose the representative
  independently of the clustering.
- [Extreme periods how-to](../how-to/extreme_periods.ipynb) — keep the peak without giving
  up the J-minimising partition.